In [1]:
# Dependencies:
# pip install numpy pandas scikit-learn matplotlib torch

from pathlib import Path
from urllib.parse import unquote
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
    roc_curve,
    auc
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Configuration

In [2]:
DATASET_DIR        = Path("./Datasets")
ALL_DATASETS       = [1, 2, 3, 4]
OUTPUT_DIR         = Path("./outputs")
PRETRAINED_DIR     = Path("./cnnModels")
RANDOM_STATE       = 35
DO_TRAINING        = False
FILTER_SIZES       = [3, 4, 5, 7, 9]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_FILTERS = 64

SECURITY_HEADERS = [
    "Cookie",
    "X-Forwarded-For",
    "Referer",
    "User-Agent",
    "Authorization",
    "X-Api-Key",
]

COLORS = ["steelblue", "darkorange", "seagreen", "mediumpurple"]

# Data Pipelines

In [3]:
def loadAndFlatten(path):
    raw = pd.read_json(path)
    df  = pd.json_normalize(raw["request"])
    if "Attack_Tag" in df.columns:
        df["label"]       = df["Attack_Tag"].apply(
            lambda x: "Malicious" if pd.notna(x) and str(x).strip() != "" else "Benign"
        )
        df["attack_type"] = df["Attack_Tag"].fillna("Benign")
    else:
        df["label"]       = "Benign"
        df["attack_type"] = "Benign"
    df["_headers_raw"] = raw["request"].apply(
        lambda r: r.get("headers", {}) if isinstance(r, dict) else {}
    )
    return df

def buildText(row):
    parts = []

    for col in ["method", "url", "body"]:
        val = row.get(col, "")
        if pd.notna(val) and str(val).strip():
            parts.append(str(val))

    headers = row.get("_headers_raw", {})
    if isinstance(headers, dict):
        for hname in SECURITY_HEADERS:
            hval = headers.get(hname, "")
            if hval and str(hval).strip():
                parts.append(f"{hname.lower()} {str(hval)}")

    text = " ".join(parts)
    try:
        text = unquote(text)
    except Exception:
        pass
    return text.lower().strip()

def prepareDataForPytorch(texts, labels, maxLen=2048):
    chars = sorted(list(set("".join(texts))))
    charToInt = {c: i+1 for i, c in enumerate(chars)}

    encoded = []
    for t in texts:
        seq = [charToInt.get(c, 0) for c in t[:maxLen]]
        seq += [0] * (maxLen - len(seq))
        encoded.append(seq)

    X = torch.tensor(encoded, dtype=torch.long)
    y = torch.tensor(labels, dtype=torch.float32).view(-1, 1)
    return X, y, len(chars) + 1, charToInt

def encodeFromMapping(texts, charToInt, maxLen=2048):
    encoded = []
    for t in texts:
        seq = [charToInt.get(c, 0) for c in t[:maxLen]]
        seq += [0] * (maxLen - len(seq))
        encoded.append(seq)

    return torch.tensor(encoded, dtype=torch.long)

# CNN Class Definition

In [4]:
class CNN(nn.Module):
    def __init__(self, numEmbeddings, embeddingDimension, numFilters, filterSizes):
        super(CNN, self).__init__()

        self.embedding = nn.Embedding(numEmbeddings, embeddingDimension)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embeddingDimension,
                      out_channels=numFilters,
                      kernel_size=filterSize)
            for filterSize in filterSizes
        ])

        self.fc = nn.Linear(len(filterSizes) * numFilters, 1)
        self.dropout = nn.Dropout(0.3)

    def forward(self, text):
        embeddedText = self.embedding(text).permute(0, 2, 1)

        conved = [torch.relu(conv(embeddedText)) for conv in self.convs]
        pooled = [torch.max(c, dim=2)[0] for c in conved]

        cat = self.dropout(torch.cat(pooled, dim=1))
        return self.fc(cat)

# Training and Testing

In [5]:
def test(loader, model, yTest):
    model.eval()
    allLogits = []

    with torch.no_grad():
        for batchX, _ in loader:
            batchX = batchX.to(DEVICE)
            logits = model(batchX)
            allLogits.append(logits)

    logitsCat = torch.cat(allLogits, dim=0)
    probabilities = torch.sigmoid(logitsCat).detach().cpu().numpy()
    predictions = (probabilities > 0.5).astype(int)

    accuracy = accuracy_score(yTest, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(yTest, predictions, average="macro")
    confMat = confusion_matrix(yTest, predictions)
    fpr, tpr, _ = roc_curve(yTest, probabilities)
    aucValue = auc(fpr, tpr)

    results = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

    return results, confMat, (fpr, tpr), aucValue

def trainAndTest(datasetPath, datasetNum):
    df = loadAndFlatten(datasetPath)
    texts = df.apply(buildText, axis=1)
    le     = LabelEncoder()
    labels = le.fit_transform(df["label"])
    labelNames = [str(c) for c in le.classes_]
    
    XTrainRaw, XTestRaw, yTrainRaw, yTestRaw = train_test_split(
        texts, labels, test_size=0.20, random_state=RANDOM_STATE, stratify=labels
    )
    XTrain, yTrain, numEmbeddings, charToInt = prepareDataForPytorch(XTrainRaw, yTrainRaw)
    XTest = encodeFromMapping(XTestRaw, charToInt)
    yTest = torch.tensor(yTestRaw, dtype=torch.float32).view(-1, 1)
    
    trainDataset = TensorDataset(XTrain, yTrain)
    testDataset = TensorDataset(XTest, yTest)
    
    trainLoader = DataLoader(trainDataset, batch_size=32, shuffle=True)
    testLoader = DataLoader(testDataset, batch_size=32)
    
    model = CNN(numEmbeddings=numEmbeddings, embeddingDimension=64, numFilters=NUM_FILTERS, filterSizes=FILTER_SIZES)
    model.to(DEVICE)
    optimizer = optim.Adam(model.parameters())
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.Tensor([10.0]).to(DEVICE))
    
    for epoch in range(5):
        model.train()
        for batchX, batchY in trainLoader:
            batchX = batchX.to(DEVICE)
            batchY = batchY.to(DEVICE)
            optimizer.zero_grad() # reset gradients
            predictions = model(batchX) # forward pass
            loss = criterion(predictions, batchY)
            loss.backward() # backwards pass
            optimizer.step()
        print(f"Epoch {epoch+1} complete for {datasetPath}.")

    modelName = f"cnn{datasetNum}"
    torch.save(model.state_dict(), PRETRAINED_DIR / f"{modelName}_weights.pth")
    artifacts = {
        "charToInt": charToInt,
        "numEmbeddings": numEmbeddings,
    }
    torch.save(artifacts, PRETRAINED_DIR / f"{modelName}_artifacts.pth")

    results = test(testLoader, model, yTest)
    return results

def loadModelAndTest(datasetPath, datasetNum):
    modelName = f"cnn{datasetNum}"
    artifacts = torch.load(PRETRAINED_DIR / f"{modelName}_artifacts.pth", map_location=DEVICE)
    weights = torch.load(PRETRAINED_DIR / f"{modelName}_weights.pth", map_location=DEVICE)

    model = CNN(
        artifacts["numEmbeddings"],
        64,
        NUM_FILTERS,
        FILTER_SIZES
    )
    model.to(DEVICE)
    model.load_state_dict(weights)
    model.eval()
    df = loadAndFlatten(datasetPath)
    texts = df.apply(buildText, axis=1)
    le     = LabelEncoder()
    labels = le.fit_transform(df["label"])
    labelNames = [str(c) for c in le.classes_]

    _, XTestRaw, _, yTestRaw = train_test_split(texts, labels, test_size=0.20, random_state=RANDOM_STATE, stratify=labels)
    
    encodedTexts = encodeFromMapping(XTestRaw, artifacts['charToInt'])
    labelsTensor = torch.tensor(yTestRaw, dtype=torch.float32).view(-1, 1)
    testDataset = TensorDataset(encodedTexts, labelsTensor)
    testLoader = DataLoader(testDataset, batch_size=32, shuffle=False)
    return test(testLoader, model, yTestRaw)

# Plots

In [6]:
def plotConfusionMatrices(allResults, modelType):
    n   = len(allResults)
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()

    for i, result in enumerate(allResults):
        disp = ConfusionMatrixDisplay(confusion_matrix=result["test_cm"],
                                      display_labels=result["label_names"])
        disp.plot(cmap="Blues", ax=axes[i], colorbar=False)
        axes[i].set_title(f"Dataset {result["dataset"]}", fontsize=12)

    for j in range(n, 4):
        axes[j].set_visible(False)

    fig.suptitle(f"Confusion Matrices — {modelType}",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / modelType / "cm_test.png", dpi=120)
    plt.close()


def plotMetrics(allResults, modelType):
    datasets    = [f"DS {r['dataset']}" for r in allResults]
    metrics     = ["accuracy", "precision", "recall", "f1"]
    metricNames = ["Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)"]
    x     = np.arange(len(datasets))
    width = 0.5

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for i, (metric, name) in enumerate(zip(metrics, metricNames)):
        vals = [r["test_metrics"][metric] for r in allResults]
        bars = axes[i].bar(x, vals, width, color=COLORS[:len(vals)], alpha=0.85)

        for bar, v in zip(bars, vals):
            axes[i].text(bar.get_x() + bar.get_width() / 2, v + 0.003,
                         f"{v:.3f}", ha="center", fontsize=9, fontweight="bold")

        axes[i].set_title(name)
        axes[i].set_xticks(x)
        axes[i].set_xticklabels(datasets)
        axes[i].set_ylim(max(0, min(vals) - 0.05), 1.05)
        axes[i].set_ylabel("Score")
        axes[i].grid(axis="y", alpha=0.3)

    fig.suptitle(f"{modelType} — Test Metrics Across All Datasets",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / modelType / "metrics.png", dpi=120, bbox_inches="tight")
    plt.close()


def plotROCCurves(allResults, modelType):
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    for colIdx, r in enumerate(allResults):
        ax = axes[colIdx]
        fpr, tpr = r["roc"]
        aucVal   = r["auc"]

        ax.plot(fpr, tpr, color="darkorange", linewidth=2,
                label=f"AUC = {aucVal:.4f}")
        ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5)
        ax.fill_between(fpr, tpr, alpha=0.12, color="darkorange")
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1.02])
        ax.set_xlabel("False Positive Rate", fontsize=9)
        ax.set_ylabel("True Positive Rate", fontsize=9)
        ax.set_title(f"Dataset {r['dataset']}", fontsize=10)
        ax.legend(loc="lower right", fontsize=10)
        ax.grid(alpha=0.3)

    fig.suptitle(f"ROC Curves - {modelType}",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / modelType / "roc_curves.png", dpi=120, bbox_inches="tight")
    plt.close()

# Main

In [7]:
def main():
    allResults = []

    if DO_TRAINING:
        print("Training and Testing Model...")
        for dataset in ALL_DATASETS:
            results, confMat, rocTuple, aucValue = trainAndTest(DATASET_DIR / f"dataset_{dataset}_train.json", dataset)
            allResults.append({
                "dataset": dataset,
                "label_names": ["Benign", "Malicious"],
                "test_metrics": results,
                "test_cm": confMat,
                "roc": rocTuple,
                "auc": aucValue
            })
    else:
        print("Testing pretrained models...")
        for dataset in ALL_DATASETS:
            results, confMat, rocTuple, aucValue = loadModelAndTest(DATASET_DIR / f"dataset_{dataset}_train.json", dataset)
            allResults.append({
                "dataset": dataset,
                "label_names": ["Benign", "Malicious"],
                "test_metrics": results,
                "test_cm": confMat,
                "roc": rocTuple,
                "auc": aucValue
            })

    print("Plotting Results...")
    for result in allResults:
        plotROCCurves(allResults, "CNN")
        plotMetrics(allResults, "CNN")
        plotConfusionMatrices(allResults, "CNN")

    print(f"All outputs saved to: {OUTPUT_DIR.resolve()}")

main()

Testing pretrained models...
Plotting Results...
All outputs saved to: /home/cirick/school/aiEnhancedSecurity/finalProject/outputs


# Demo

In [15]:
def loadModel(dataset):
    modelName = f"cnn{dataset}"
    artifacts = torch.load(PRETRAINED_DIR / f"{modelName}_artifacts.pth", map_location=DEVICE)
    weights = torch.load(PRETRAINED_DIR / f"{modelName}_weights.pth", map_location=DEVICE)

    model = CNN(
        numEmbeddings=artifacts["numEmbeddings"],
        embeddingDimension=64,
        numFilters=NUM_FILTERS,
        filterSizes=FILTER_SIZES
    )

    model.load_state_dict(weights)
    model.to(DEVICE)
    model.eval()

    return model, artifacts['charToInt']

def predictRequest(model, charToInt, method, url, body, headers):
    request = {
        "method": method,
        "url": url,
        "body": body,
        "_headers_raw": headers
    }
    rawText = buildText(request)

    inputTensor = encodeFromMapping([rawText], charToInt, maxLen=2048)
    inputTensor = inputTensor.to(DEVICE)

    with torch.no_grad():
        logits = model(inputTensor)
        probability = torch.sigmoid(logits).item()

    result = "Malicious" if probability > 0.5 else "Benign"
    return result

model, charToInt = loadModel(4)

from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import uvicorn

app = FastAPI()

@app.middleware("http")
async def middleware(request: Request, call_next):
    body = await request.body()
    requestDict = {
        "method": request.method,
        "url": str(request.url),
        "body": body.decode(),
        "_headers_raw": dict(request.headers)
    }

    prediction = predictRequest(model, charToInt, requestDict['method'], requestDict['url'], requestDict['body'], requestDict['_headers_raw'])

    if prediction == "Malicious":
        return JSONResponse(
        status_code=403,
        content={
            "message": "Malicious Request"
        }
    )

    return await call_next(request)

@app.get("/api/data")
async def normalRoute():
    return JSONResponse(
        status_code=200,
        content={
            "message": "Benign Request"
        }
    )

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [1375]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:55149 - "GET /api/data HTTP/1.1" 200 OK
INFO:     127.0.0.1:55155 - "GET /api/data?id=SELECT%20*%20FROM%20users%20WHERE%20admin%20=%20'true' HTTP/1.1" 403 Forbidden


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1375]
